# In-Memory Vector Store with Gemini Embeddings and Groq

**A beginner-friendly, interview-ready notebook based on the instructor's `3-Othervectorstores.ipynb` and the lesson transcript.**

We will store ten small `Document` objects in LangChain's `InMemoryVectorStore`, retrieve semantically similar documents, inspect scores, filter by metadata, convert the store into a retriever, and connect it to a basic RAG pipeline.

**Provider roles are separate:**
- **Google Gemini API** creates embeddings used by the vector store.
- **Groq API** runs the chat model that writes answers from retrieved context.
- **LangChain Core** supplies `Document`, `Embeddings`, `InMemoryVectorStore`, retrievers, prompts, and LCEL composition.

> Terminology note: the instructor's code uses `ChatGroq` and `GROQ_API_KEY`. That is **Groq**, not xAI's separate **Grok** service. This notebook follows the uploaded course source and therefore uses Groq.

## What you should be able to explain after this notebook

1. Define an in-memory vector store and explain how it stores documents and vectors.
2. Explain embeddings, cosine similarity, top-*k*, metadata filters, and retrievers.
3. Implement `similarity_search`, `similarity_search_by_vector`, and `similarity_search_with_score`.
4. Explain why this store is useful for learning and prototyping, and why it is not a replacement for a persistent vector database at scale.
5. Build a basic RAG chain that uses Gemini for retrieval embeddings and Groq for generation.

**Source fidelity:** the ten sample documents, `add_documents`, the weather-search example, `k=2`, `as_retriever()`, and `retriever.invoke()` are preserved from the instructor notebook. The instructor's OpenAI chat/embedding setup is replaced with separate Gemini and Groq setup cells, as requested. The transcript mentions filtering and building an LCEL/helper-based RAG pipeline; those are added as explicitly explained demonstrations. The blank final source cell is omitted.

## 1. Environment setup

Run these in **Windows CMD** from your project directory, for example `E:\RAG`:

```cmd
uv add python-dotenv google-genai langchain-core langchain-groq numpy
uv add --dev jupyter ipykernel
```

If the project environment is already configured, add only missing dependencies. Do **not** run `uv run -r requirements.txt`; `uv run` runs a command, it is not the requirements-file installation command.

Create a `.env` file in the project root (never commit it to Git):

```text
GEMINI_API_KEY=your_gemini_api_key
GROQ_API_KEY=your_groq_api_key
```

This notebook makes real API requests in the provider smoke tests, embedding adapter tests, indexing, and RAG queries. Execute those cells only when you are ready to use your API quota.

## 2. Shared imports and environment loading

This cell imports provider-neutral LangChain and Python utilities only. Gemini's SDK and Groq's chat integration are imported later in **separate provider sections**, so it is clear which API each cell is using.

In [1]:
import os
from pathlib import Path

import numpy as np
from dotenv import find_dotenv, load_dotenv

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore

# Search from the working directory, which is convenient in Jupyter.
env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path, override=False)

print(f".env file found: {bool(env_path)}")
print(f"Working directory: {Path.cwd()}")

.env file found: True
Working directory: e:\RAG\Vector_stores_&_Vector_databases


## 3. What is `InMemoryVectorStore`?

A **vector store** keeps text records (usually `Document` objects), their embeddings, and a way to retrieve records by vector similarity. LangChain's `InMemoryVectorStore` stores its records in a Python dictionary and computes cosine similarity for search using NumPy.

### Why and when to use it

- **Good for:** tutorials, quick prototypes, unit tests, and small datasets where simple setup matters.
- **Not ideal for:** large corpora, multi-user services, durable storage, distributed search, or operationally demanding production systems.
- **Important limitation:** the contents live in the current Python process. Restarting the kernel or process normally loses the populated in-memory store unless you explicitly export and reload it.

A persistent system such as Chroma or a hosted vector database addresses different operational needs. FAISS is a search/index library; it also requires you to manage index persistence and associated document metadata. These tools are alternatives, not interchangeable storage guarantees.

## 4. Provider A — Gemini embeddings (retrieval only)

An **embedding** is a numerical vector produced from text. Similar meanings often map to nearby vector representations. Here Gemini does **not** generate the final RAG answer; it converts document text and search queries into vectors so `InMemoryVectorStore` can compare them.

We use the current Gemini Embedding 2 model ID shown in Google's documentation: `gemini-embedding-2`. For text retrieval, Google recommends consistent task formatting: `title: ... | text: ...` for documents and `task: search result | query: ...` for queries. We follow that formatting throughout this adapter.

Official reference: [Gemini embeddings](https://ai.google.dev/gemini-api/docs/embeddings).

In [2]:
from google import genai

if not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError(
        "GEMINI_API_KEY is missing. Add it to your .env file, then rerun this cell."
    )

GEMINI_EMBEDDING_MODEL = "gemini-embedding-2"
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print(f"Gemini embedding model configured: {GEMINI_EMBEDDING_MODEL}")

Gemini embedding model configured: gemini-embedding-2


### 4.1 Test Gemini independently

Before connecting the model to LangChain, make one direct embedding request. This isolates API-key, model-access, network, and response-shape issues from vector-store bugs. The returned vector length is printed dynamically; do not hardcode dimensions because embedding models can expose different output dimensions.

In [3]:
sample_embedding_response = gemini_client.models.embed_content(
    model=GEMINI_EMBEDDING_MODEL,
    contents="task: search result | query: What is semantic search?",
)

if not sample_embedding_response.embeddings:
    raise RuntimeError("Gemini returned no embeddings for the smoke test.")

sample_vector = sample_embedding_response.embeddings[0].values
if not sample_vector:
    raise RuntimeError("Gemini returned an empty vector for the smoke test.")

print("Embedding dimension:", len(sample_vector))
print("First five values:", [round(float(value), 5)
      for value in sample_vector[:5]])

Embedding dimension: 3072
First five values: [0.00226, 0.01006, -0.00687, -0.01161, 0.00028]


### 4.2 Adapt Gemini to LangChain's `Embeddings` interface

LangChain vector stores expect an embedding object with two methods:

- `embed_documents(texts)` returns one vector per document string.
- `embed_query(text)` returns one vector for a single search query.

The custom adapter below implements that interface using the Gemini SDK. To keep behavior unambiguous for this beginner example, it sends one text per request instead of relying on multi-input response aggregation behavior. This is easy to understand but can be slower and use more API requests than a carefully implemented batch workflow.

Notice the different formatting in the two methods: **document formatting happens during indexing**, while **query formatting happens during search**. Use the same model and compatible embedding settings for both.

In [4]:
class GeminiEmbeddings(Embeddings):
    """Gemini Embedding 2 adapter compatible with LangChain vector stores."""

    def __init__(self, client, model: str = GEMINI_EMBEDDING_MODEL):
        self.client = client
        self.model = model

    def _embed_one(self, formatted_text: str) -> list[float]:
        response = self.client.models.embed_content(
            model=self.model,
            contents=formatted_text,
        )
        if not response.embeddings or not response.embeddings[0].values:
            raise RuntimeError("Gemini returned an empty embedding.")
        return [float(value) for value in response.embeddings[0].values]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        vectors = []
        for text in texts:
            formatted_document = f"title: none | text: {text}"
            vectors.append(self._embed_one(formatted_document))
        return vectors

    def embed_query(self, text: str) -> list[float]:
        formatted_query = f"task: search result | query: {text}"
        return self._embed_one(formatted_query)


gemini_embeddings = GeminiEmbeddings(client=gemini_client)
print("GeminiEmbeddings adapter created.")

GeminiEmbeddings adapter created.


### 4.3 Validate the adapter

This cell tests the **LangChain-facing method**, not just the direct SDK call above. If this works, `InMemoryVectorStore` can call `embed_documents()` during insertion and `embed_query()` during search.

In [5]:
adapter_test_vector = gemini_embeddings.embed_query(
    "How is tomorrow's weather expected to be?")
print("Vector type:", type(adapter_test_vector).__name__)
print("Vector dimension:", len(adapter_test_vector))
print("All values finite:", bool(np.isfinite(adapter_test_vector).all()))

Vector type: list
Vector dimension: 3072
All values finite: True


## 5. Create the instructor's ten sample documents

The instructor models each record with LangChain's `Document`:

- `page_content`: the actual text to embed and retrieve.
- `metadata`: structured information about the text; here it stores `source` (`tweet`, `news`, or `website`).

The ten examples below preserve the original texts and metadata. They are intentionally short, so this lesson does **not** apply a text splitter. In a real document-ingestion pipeline, long source files are typically parsed and chunked before embedding; for these already-small sample records, adding a splitter would change the instructor's example unnecessarily.

In [6]:
document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

print(f"Number of source documents: {len(documents)}")

Number of source documents: 10


### 5.1 Inspect the data before indexing

Always inspect a few records before sending them to an embedding model. This checks that the text and metadata are shaped as expected. It is much easier to find a source-data problem before embedding and indexing than after search results look wrong.

In [7]:
for index, document in enumerate(documents[:3], start=1):
    print(f"Document {index}")
    print("Text:", document.page_content)
    print("Metadata:", document.metadata)
    print("-" * 60)

Document 1
Text: I had chocolate chip pancakes and scrambled eggs for breakfast this morning.
Metadata: {'source': 'tweet'}
------------------------------------------------------------
Document 2
Text: The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
Metadata: {'source': 'news'}
------------------------------------------------------------
Document 3
Text: Building an exciting new project with LangChain - come check it out!
Metadata: {'source': 'tweet'}
------------------------------------------------------------


## 6. Initialize the in-memory vector store

The embedding object is passed into `InMemoryVectorStore`. The store will call it whenever it needs document vectors or a query vector. This constructor does **not** call Groq and it does not generate answers; this section is exclusively about vector storage and retrieval.

In [8]:
vector_store = InMemoryVectorStore(embedding=gemini_embeddings)
print("InMemoryVectorStore initialized with Gemini embeddings.")

InMemoryVectorStore initialized with Gemini embeddings.


### 6.1 Add the documents

`add_documents()` embeds each document and stores its text, metadata, ID, and vector. Explicit stable IDs make the example easier to inspect and make repeated insertion with the same IDs less confusing than generating unrelated IDs each run.

This call triggers Gemini embedding requests. It does not call Groq. For this small example, a simple per-text adapter is fine; for large datasets, batch strategy, rate limits, retries, and caching become important.

In [9]:
document_ids = [
    f"sample-doc-{index:02d}" for index in range(1, len(documents) + 1)]
inserted_ids = vector_store.add_documents(
    documents=documents, ids=document_ids)

print(f"Requested documents: {len(documents)}")
print(f"IDs returned by the store: {len(inserted_ids)}")
print("First three IDs:", inserted_ids[:3])

Requested documents: 10
IDs returned by the store: 10
First three IDs: ['sample-doc-01', 'sample-doc-02', 'sample-doc-03']


## 7. Semantic similarity search

The instructor's first search query is `"hows the weather forecast"`. The spelling is kept so you can line up this result with the course. The store embeds the query with Gemini and ranks stored vectors by cosine similarity.

**Cosine similarity intuition:** it compares the angle between two vectors. For this implementation, the returned score from `similarity_search_with_score()` is a similarity score; larger means more similar. Do not confuse this with FAISS examples that often show a distance where smaller is better.

In [10]:
weather_query = "hows the weather forecast"
weather_results = vector_store.similarity_search(weather_query)

print(f"Query: {weather_query!r}")
print(f"Documents returned with the default k: {len(weather_results)}")
for rank, document in enumerate(weather_results, start=1):
    print(f"\nResult {rank}")
    print("Text:", document.page_content)
    print("Metadata:", document.metadata)

Query: 'hows the weather forecast'
Documents returned with the default k: 4

Result 1
Text: The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
Metadata: {'source': 'news'}

Result 2
Text: I have a bad feeling I am going to get deleted :(
Metadata: {'source': 'tweet'}

Result 3
Text: I had chocolate chip pancakes and scrambled eggs for breakfast this morning.
Metadata: {'source': 'tweet'}

Result 4
Text: The stock market is down 500 points today due to fears of a recession.
Metadata: {'source': 'news'}


### 7.1 Control the number of results with `k`

`k` means the number of top-ranked documents requested. It does **not** mean a relevance threshold, probability, or percentage. If you request `k=2`, the store returns up to two records, ranked by similarity (assuming at least two eligible documents are available). Tune `k` using retrieval tests rather than assuming that a larger value is always better.

In [11]:
top_two_results = vector_store.similarity_search(weather_query, k=2)

print(f"Top-{len(top_two_results)} results for {weather_query!r}:")
for rank, document in enumerate(top_two_results, start=1):
    print(f"\n{rank}. {document.page_content}")
    print("   metadata:", document.metadata)

Top-2 results for 'hows the weather forecast':

1. The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
   metadata: {'source': 'news'}

2. I have a bad feeling I am going to get deleted :(
   metadata: {'source': 'tweet'}


### 7.2 Search using a query vector directly

`similarity_search()` accepts text and embeds it internally. `similarity_search_by_vector()` accepts an embedding that you already computed. This is useful when the query vector is already available from another part of a pipeline. The vector must come from the same compatible embedding space as the stored document vectors.

In [12]:
query_vector = gemini_embeddings.embed_query(weather_query)
results_by_vector = vector_store.similarity_search_by_vector(query_vector, k=2)

for rank, document in enumerate(results_by_vector, start=1):
    print(f"{rank}. {document.page_content}")
    print("   metadata:", document.metadata)

1. The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
   metadata: {'source': 'news'}
2. I have a bad feeling I am going to get deleted :(
   metadata: {'source': 'tweet'}


### 7.3 Retrieve documents with similarity scores

`similarity_search_with_score()` returns `(Document, score)` pairs. Here the in-memory implementation uses cosine similarity, so a larger score means greater vector similarity. The score is **not** a calibrated probability that the document is correct or relevant to a human.

Always identify the metric and the store implementation before interpreting a score; score semantics can differ between vector stores.

The transcript also names `similarity_search_with_relevance_scores()`. The generic LangChain `VectorStore` API has that method, but its normalization depends on each implementation's relevance-score function. The current `InMemoryVectorStore` implementation specifically documents and implements raw cosine scores for `similarity_search_with_score()`; do not assume a normalized relevance score is calibrated or supported correctly for every version. For this lesson, inspect the raw cosine scores unless you have verified the installed implementation.


In [13]:
scored_results = vector_store.similarity_search_with_score(weather_query, k=3)

for rank, (document, score) in enumerate(scored_results, start=1):
    print(f"Rank: {rank} | cosine similarity: {score:.4f}")
    print("Text:", document.page_content)
    print("Metadata:", document.metadata)
    print("-" * 60)

Rank: 1 | cosine similarity: 0.7360
Text: The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
Metadata: {'source': 'news'}
------------------------------------------------------------
Rank: 2 | cosine similarity: 0.5721
Text: I have a bad feeling I am going to get deleted :(
Metadata: {'source': 'tweet'}
------------------------------------------------------------
Rank: 3 | cosine similarity: 0.5674
Text: I had chocolate chip pancakes and scrambled eggs for breakfast this morning.
Metadata: {'source': 'tweet'}
------------------------------------------------------------


### 7.4 Filter the search space using metadata

The transcript mentions filtering, although the provided notebook does not implement it. This cell adds a working example. In the core `InMemoryVectorStore` API, `filter` is a **callable predicate** that receives a `Document` and returns `True` when the document should be eligible for search.

This is an important portability detail: other vector stores may use dictionary-style metadata filters instead. Do not assume identical filter syntax across Chroma, FAISS wrappers, and in-memory storage.

In [14]:
news_results = vector_store.similarity_search(
    weather_query,
    k=3,
    filter=lambda document: document.metadata.get("source") == "news",
)

print(f"News-only results: {len(news_results)}")
for rank, document in enumerate(news_results, start=1):
    print(f"{rank}. {document.page_content}")
    print("   metadata:", document.metadata)

News-only results: 3
1. The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
   metadata: {'source': 'news'}
2. The stock market is down 500 points today due to fears of a recession.
   metadata: {'source': 'news'}
3. Robbers broke into the city bank and stole $1 million in cash.
   metadata: {'source': 'news'}


## 8. Turn the vector store into a retriever

A **vector store** manages records and search. A **retriever** exposes a simple interface that takes a query and returns relevant `Document` objects. This abstraction lets a RAG pipeline use a retriever without needing to know every storage detail.

The instructor calls `as_retriever()` with default settings first, then sets `search_kwargs={"k": 2}`. We'll do both.

In [15]:
retriever = vector_store.as_retriever()
print(retriever)

tags=['InMemoryVectorStore', 'GeminiEmbeddings'] vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x0000023D4209B020> search_kwargs={}


### 8.1 Invoke the retriever

`retriever.invoke(query)` is the standard synchronous interface. The default number of returned documents depends on the retriever defaults; use explicit `search_kwargs` if your application requires a defined count.

In [16]:
default_retrieved_docs = retriever.invoke(weather_query)

print(f"Documents returned: {len(default_retrieved_docs)}")
for rank, document in enumerate(default_retrieved_docs, start=1):
    print(f"{rank}. {document.page_content}")
    print("   metadata:", document.metadata)

Documents returned: 4
1. The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
   metadata: {'source': 'news'}
2. I have a bad feeling I am going to get deleted :(
   metadata: {'source': 'tweet'}
3. I had chocolate chip pancakes and scrambled eggs for breakfast this morning.
   metadata: {'source': 'tweet'}
4. The stock market is down 500 points today due to fears of a recession.
   metadata: {'source': 'news'}


### 8.2 Set `k=2` with `search_kwargs`

The `search_kwargs` dictionary is passed to the underlying search function. This is why `k` belongs there when configuring a retriever. It also makes search configuration explicit and reusable.

In [17]:
top_two_retriever = vector_store.as_retriever(search_kwargs={"k": 2})
top_two_retrieved_docs = top_two_retriever.invoke(weather_query)

print(f"Top-two retriever returned: {len(top_two_retrieved_docs)} document(s)")
for rank, document in enumerate(top_two_retrieved_docs, start=1):
    print(f"{rank}. {document.page_content}")

Top-two retriever returned: 2 document(s)
1. The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
2. I have a bad feeling I am going to get deleted :(


### 8.3 Apply a metadata filter through the retriever

Because the underlying in-memory store accepts a callable `filter`, the same predicate can be passed through `search_kwargs`. This keeps the retrieval configuration together. Confirm this behavior against the vector-store integration you use in another project; not all integrations use the same filter API.

In [18]:
news_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": lambda document: document.metadata.get("source") == "news",
    }
)

news_retrieved_docs = news_retriever.invoke(weather_query)
print(f"News-only retriever returned: {len(news_retrieved_docs)} document(s)")
for document in news_retrieved_docs:
    print(document.metadata.get("source"), "—", document.page_content)

News-only retriever returned: 3 document(s)
news — The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
news — The stock market is down 500 points today due to fears of a recession.
news — Robbers broke into the city bank and stole $1 million in cash.


## 9. Provider B — Groq chat model (answer generation only)

This section is separate from Gemini. Groq does **not** create the document or query embeddings in this notebook; it generates text after the retrieval stage supplies context.

The course transcript's speech-to-text sometimes says “Grok,” but the actual integration in the source uses `ChatGroq` and `GROQ_API_KEY`. This notebook follows that source. The model ID below is `openai/gpt-oss-120b`, which is a model served by Groq's API; the `openai/` prefix identifies the model family, not the API provider.

Official reference: [Groq + LangChain](https://console.groq.com/docs/langchain) and [Groq supported models](https://console.groq.com/docs/models).

In [19]:
from langchain_groq import ChatGroq

if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError(
        "GROQ_API_KEY is missing. Add it to your .env file, then rerun this cell."
    )

GROQ_MODEL = "openai/gpt-oss-120b"
groq_llm = ChatGroq(model=GROQ_MODEL, temperature=0)

print(f"Groq chat model configured: {GROQ_MODEL}")

Groq chat model configured: openai/gpt-oss-120b


### 9.1 Test Groq independently

Test the generation provider before building the combined RAG chain. This makes debugging clearer: if this cell fails, investigate the Groq key, model access, quota, or network before blaming the vector store. This request sends no Gemini embedding request.

In [20]:
groq_test_response = groq_llm.invoke(
    "In one or two sentences, explain what a vector store does."
)
print(groq_test_response.content)

A vector store is a database that indexes high‑dimensional vectors (embeddings) and enables fast similarity search, letting you retrieve items whose vectors are nearest to a query vector. It’s used to match text, images, or other data based on semantic similarity.


## 10. Connect retrieval to generation: a basic RAG pipeline

**Retrieval-Augmented Generation (RAG)** retrieves potentially relevant information and passes it to a language model as context. This helps the model answer using the supplied knowledge instead of relying only on its parametric training.

The provider setup remains in its own section. The following cells only define the prompt and composition. The actual RAG invocation is an intentional integration point: retrieval embeds the question with Gemini, then Groq generates the answer from the retrieved text. A complete RAG query must connect these two roles somewhere.

### 10.1 Create the answer prompt

The prompt has two variables:

- `{context}` — text from retrieved documents.
- `{question}` — the user's question.

The prompt instructs the model to use only the available context and to say when the context is insufficient. This reduces unsupported answers, but it is not a guarantee against hallucination; retrieval and answer quality still need evaluation.

In [21]:
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a careful question-answering assistant. "
            "Answer using only the supplied context. If the context does not "
            "contain enough information, say that you do not know from the provided context.\n\n"
            "Context:\n{context}",
        ),
        ("human", "Question: {question}"),
    ]
)
print("RAG prompt created.")

RAG prompt created.


### 10.2 Format retrieved documents into context

The retriever returns a list of `Document` objects. The prompt expects a string. `format_docs()` converts the list into readable text and includes the `source` metadata so the LLM receives a useful provenance hint. Metadata shown in a prompt is not by itself a citation-verification system.

In [22]:
def format_docs(docs: list[Document]) -> str:
    """Join retrieved document text and source labels into prompt context."""
    formatted_documents = []
    for document in docs:
        source = document.metadata.get("source", "unknown")
        formatted_documents.append(
            f"Source: {source}\nContent: {document.page_content}"
        )
    return "\n\n---\n\n".join(formatted_documents)

### 10.3 Build the LCEL chain

LCEL (`LangChain Expression Language`) connects runnables using `|`.

1. The mapping sends the question to the retriever and to the `question` field.
2. The retriever returns documents; `RunnableLambda(format_docs)` turns them into context text.
3. The prompt formats the context and question.
4. `groq_llm` generates a response.
5. `StrOutputParser` extracts a plain string.

This is the notebook's main integration cell. It does not create clients or set API keys; those provider-specific steps were completed separately above.

In [23]:
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | groq_llm
    | StrOutputParser()
)

print("LCEL RAG chain assembled. No generation has run yet.")

LCEL RAG chain assembled. No generation has run yet.


### 10.4 Ask a question through RAG

This invocation triggers both stages: Gemini embeds the question for retrieval; Groq generates the answer from the retrieved documents. The source data only says tomorrow's forecast is cloudy/overcast with a high of 62 degrees, so the answer should stay within that evidence.

In [24]:
rag_question = "How is the weather forecast for tomorrow?"
rag_answer = rag_chain.invoke(rag_question)
print("Question:", rag_question)
print("Answer:", rag_answer)

Question: How is the weather forecast for tomorrow?
Answer: The forecast calls for tomorrow to be cloudy and overcast, with a high temperature of about 62 °F.


### 10.5 Inspect the evidence separately

For debugging, evaluate the retriever independently as well as the final answer. If the answer is wrong because the wrong records were retrieved, prompt changes alone will not fix the retrieval failure. This cell uses the Gemini-backed retriever only and does not call Groq.

In [25]:
rag_evidence = retriever.invoke(rag_question)
for rank, document in enumerate(rag_evidence, start=1):
    print(f"Source result {rank}: {document.metadata}")
    print(document.page_content)
    print("-" * 60)

Source result 1: {'source': 'news'}
The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
------------------------------------------------------------
Source result 2: {'source': 'tweet'}
I had chocolate chip pancakes and scrambled eggs for breakfast this morning.
------------------------------------------------------------
Source result 3: {'source': 'tweet'}
I have a bad feeling I am going to get deleted :(
------------------------------------------------------------
Source result 4: {'source': 'news'}
The stock market is down 500 points today due to fears of a recession.
------------------------------------------------------------


## 11. Optional alternative: helper-based retrieval chain

The transcript also mentions `create_stuff_documents_chain()` and `create_retrieval_chain()`. They package document formatting and retrieval/generation into higher-level helpers. The main implementation above uses LCEL directly to show each step explicitly.

**Optional dependency:** the current helper functions are in `langchain-classic`. If you want to run the alternative cells, add it in CMD from your project folder:

```cmd
uv add langchain-classic
```

This route is included for course compatibility and comparison; it is not required to run the primary LCEL chain. The helper setup uses the already-configured Groq model and the already-configured in-memory retriever.

### 11.1 Import the optional helpers

This cell only imports helpers. If you have not installed `langchain-classic`, skip this optional section and continue to the recap.

In [26]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

### 11.2 Create the helper-compatible prompt and chains

`create_stuff_documents_chain()` expects a prompt variable named `context` by default. `create_retrieval_chain()` takes a retriever and the document-combination chain; invocation uses a dictionary with an `input` key.

In [27]:
helper_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer only from the context below. If the answer is not present, "
            "say you do not know from the provided context.\n\nContext:\n{context}",
        ),
        ("human", "Question: {input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(groq_llm, helper_prompt)
helper_rag_chain = create_retrieval_chain(retriever, question_answer_chain)
print("Optional helper-based RAG chain assembled.")

Optional helper-based RAG chain assembled.


### 11.3 Invoke the helper-based alternative

The result dictionary includes the input and retrieved context along with an `answer` field. This is useful when you prefer a higher-level abstraction. Compare it conceptually with the explicit LCEL chain above; do not create both chains in one production path unless you actually need both.

In [28]:
helper_result = helper_rag_chain.invoke(
    {"input": "How is the weather forecast for tomorrow?"})
print("Answer:", helper_result["answer"])
print("Retrieved document count:", len(helper_result.get("context", [])))

Answer: The forecast for tomorrow calls for cloudy, overcast conditions with a high temperature of about 62 °F.
Retrieved document count: 4


## 12. When to use InMemoryVectorStore vs. other stores

| Option | Good fit | Key trade-off |
|---|---|---|
| `InMemoryVectorStore` | Learning, tests, small data, quick prototypes | Data is process-local by default; simple search scans stored vectors. |
| FAISS | Efficient local nearest-neighbor indexing and larger local vector collections | You manage persistence, metadata/doc mapping and deployment behavior. |
| Chroma | Local persistent storage and metadata-aware retrieval workflows | Adds a database/client lifecycle and persistence/configuration choices. |
| Hosted vector database | Shared services, managed operations, larger workloads | Network dependency, cost, access control, and provider-specific behavior. |

Do not choose a store based on brand popularity alone. Decide based on dataset size, latency, persistence, filtering, concurrency, operational ownership, deployment model, and evaluation results.

## 13. Source coverage and changes from the instructor notebook

The original notebook has 11 cells (including an empty final cell). The completed coverage is:

| Instructor source cell | Original purpose | Destination in this notebook |
|---:|---|---|
| 0 | InMemoryVectorStore definition | Sections 3–4: definition and setup |
| 1 | OpenAI chat model setup | Replaced by separate Groq setup and smoke test in Section 9 |
| 2 | OpenAI embeddings + vector store initialization | Replaced by Gemini adapter, then store initialization in Sections 4 and 6 |
| 3 | Ten `Document` objects and metadata | Preserved in Section 5, including all ten original texts and source labels |
| 4 | Display the documents | Replaced by readable inspection of the sample records |
| 5 | `add_documents()` | Preserved with stable explicit IDs in Section 6.1 |
| 6 | Default `similarity_search()` | Preserved in Section 7 |
| 7 | `similarity_search(..., k=2)` | Preserved in Section 7.1 |
| 8 | Convert store with `as_retriever(search_kwargs={"k": 2})` | Preserved and demonstrated with default and `k=2` retrievers in Section 8 |
| 9 | `retriever.invoke()` | Preserved in Sections 8.1–8.3 |
| 10 | Empty cell | Removed |

The transcript additionally explains the dictionary-backed implementation, cosine similarity, filtering, and that the retriever can feed an LCEL or helper-based RAG chain. Those topics are explained and demonstrated above. The notebook uses two separate provider-configuration cells and separate provider smoke tests. Only the explicitly labelled RAG composition/invocation cells connect retrieval and generation, because that connection is the definition of a RAG pipeline.

## 14. Interview-ready recap

**What is an in-memory vector store?** A vector store that holds documents and their embedding vectors in process memory and retrieves records based on vector similarity.

**Why use embeddings?** They represent text as numeric vectors so semantically related questions and documents can match even when they do not share exact keywords.

**What does `k` mean?** The requested number of top results—not a score threshold.

**What is the difference between a vector store and a retriever?** The vector store manages and searches records; the retriever exposes a standard query-to-documents interface that can plug into a chain.

**Why store metadata?** To keep structured attributes such as source, topic, tenant, or timestamp attached to a document and support filtering or source inspection.

**Why is this not automatically production-ready?** It is process-local, not designed as a durable shared database, and a simple in-memory similarity search becomes expensive as the number of vectors grows.

**Why separate Gemini and Groq?** Gemini creates the vectors for retrieval; Groq generates the response. They serve distinct parts of the RAG architecture, making debugging and provider replacement clearer.

### What to practise without notes

- Write a `Document` with metadata.
- Explain and implement `embed_documents()` vs. `embed_query()`.
- Add documents to the store and retrieve the top two.
- Explain the score returned by the in-memory implementation.
- Filter documents by metadata using a callable predicate.
- Convert a store into a retriever.
- Draw and explain: question → Gemini embedding/retrieval → retrieved context → Groq generation → answer.

### Official documentation

- [LangChain InMemoryVectorStore reference](https://reference.langchain.com/python/langchain-core/vectorstores/in_memory/InMemoryVectorStore)
- [LangChain vector-store integration guide](https://docs.langchain.com/oss/python/integrations/vectorstores)
- [Gemini embedding documentation](https://ai.google.dev/gemini-api/docs/embeddings)
- [Groq + LangChain](https://console.groq.com/docs/langchain)
- [Groq supported models](https://console.groq.com/docs/models)
- [LangChain Classic: create_stuff_documents_chain](https://reference.langchain.com/python/langchain-classic/chains/combine_documents/stuff/create_stuff_documents_chain)
- [LangChain Classic: create_retrieval_chain](https://reference.langchain.com/python/langchain-classic/chains/retrieval/create_retrieval_chain)

**Runtime reminder:** syntax validation does not prove that live API calls succeed. Run the notebook in your own `uv` environment, with valid keys and an available model. If a cell fails, fix and rerun that cell and its downstream dependants in order.